# 🎯 [PRC01_M04] LLM/자연어처리 기반 독소 조항, 납기 지체 위험 문구 자동 식별 모델 고도화 (훈련생 실습용)
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/PRC01/prc01_m04_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 조달·계약·국산화 (Level 3) |
| **미션 ID** | PRC01_M04 |
| **미션 주제** | LLM/자연어처리 기반 독소 조항, 납기 지체 위험 문구 자동 식별 모델 고도화 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
조달계획안과 계약서는 지체상금률, 선금, 보증, 하도급, 필수 조항 같은 항목이 규정에 맞는지 사람이 한 건씩 읽고 대조해야 해서 검토가 늦어지고 불공정 조항이나 납기 지체 위험 문구를 놓치기 쉽다. 본 프로젝트는 조달계획안과 계약서의 오류와 위험을 검증하는 도구를 만드는 과제로, 조달 업무의 병목과 위험 신호 기준, 조항 수정 대안, 계약 의뢰 근거와 변경 이력, 업무 효과 측정까지 한 흐름으로 구현한다. 교육용 가상 계약 자료로 계약 데이터 스키마 정의, 서식별 자료 정제와 조항 정규화, 룰 기반 규정 준수 판별, 자연어처리 기반 독소·납기 위험 문구 식별, 위험 점수와 신뢰도 엔진, 조항 하이라이트 대시보드와 오류 처리, 계약관·법무 검토 HITL 워크플로우, 1,000건 일괄 스크리닝 부하 시험, AAR 룰셋 재보정, 통합 패키지까지 단계별로 만든다. 최종 목표는 위험 계약을 근거와 함께 먼저 보여 주어 검토 시간을 줄이되, 규정 위반 여부의 판단과 계약 승인은 계약관·법무·승인권자가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

### 📁 프로젝트 표준 산출물 디렉터리 체계 (`docs/`, `src/`, `dist/`) 안내
> **💡 [실무 프로토타입 업무 도구 구축 가이드]**
> 본 프로젝트는 단순 일회성 데이터 분석 실습이 아닌, 실제 군 부서(군수사령부, 조달본부, 계약관리과 등)에 배포 가능한 **업무 자동화 소프트웨어 도구(Tool)**를 단계별로 조립·완성하는 과정입니다. 실습 중 산출물(Deliverables)로 요구되는 파일들은 아래 표준 디렉터리에 모듈별로 저장·누적됩니다.
> - **`docs/` (국방 행정 및 기술 명세서):** 업무 정의서(`prc01_as_is_to_be_spec.md`), 스키마 명세서(`schema_definition.md`), UAT 검증 보고서(`prc01_uat_feedback_report.md`), 7개 장 운영 매뉴얼(`docs/prc01_final_manual.md`)
> - **`src/` (업무 도구 소스코드):** 데이터 파서(`connectors/`), 규정 준수 룰 엔진(`engine/`), NLP 지능 모듈(`ai/`), 대시보드 UI(`ui/`), 결함 검증기(`utils/`), HITL 워크플로우(`workflow/`), 배치 러너(`pipeline/`)
> - **`dist/` (배포 패키지):** 원클릭 자동 설치 스크립트(`dist/prc01_package/setup.sh`), 설치 가이드(`README.md`)
> *(주피터 노트북 환경에서는 각 미션 셀 검증 후 `%%writefile src/...` 매직 명령어를 활용하여 해당 모듈 파일로 자동 추출할 수 있습니다.)*

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 04. LLM/자연어처리 기반 독소 조항, 납기 지체 위험 문구 자동 식별 모델 고도화
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

자연어처리 기반 독소 조항 및 납기 지체 위험 문구 식별 모델을 고도화한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **자연어처리(글자 n-gram TF-IDF와 로지스틱 회귀)로 독소 조항과 납기 지체 위험 문구를 식별하는 모델을 만들고 수정 권고 문안을 제시한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **조항 문장을 정상·독소·납기위험 세 종류로 분류하는 모델을 학습하고, 학습에 쓴 문장 틀과 처음 보는 표현 두 시험 세트로 정확도를 계산해 단순 키워드 기준선과 비교 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/ai/prc01_intelligence_module.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 추천 사유의 합리성 및 실무 수용성

#### 미션 PASS 조건

- 정량 기준: 추천/분류 정확도 85% 이상 달성


- 진행 기준: AI 추론 모듈 구현 및 벤치마크 테스트 완료


- 검증 방법: 과거 승인/처리 완료 데이터와의 추천 일치도 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""조항 문장을 정상·독소 조항·납기 지체 위험 문구로 분류하는 글자 n-gram TF-IDF 로지스틱 회귀 모델을 만들어 처음 보는 표현까지 정확도 85% 이상을 확인하고, 단순 키워드 기준선과 비교하며, 위험 조항마다 근거 표현과 표준 문안 Top-3 수정 권고(유사도 점수)를 제시한다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import make_pipeline

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 조달계획안·계약서 생성기 (실제 계약·업체·법령 조문 없음)
# 계약 속성 16개 + 조항 문장 6개를 만들고, 속성 위반·독소 조항·납기 위험 문구가 섞인 '규정 위반 여부'를 함께 만든다.
# ---------------------------------------------------------------------------
TYPES = ["물품", "용역", "공사"]
METHODS = ["일반경쟁", "제한경쟁", "지명경쟁", "수의계약"]
CLASSES = ["정상", "독소", "납기위험"]
OBJ = ["납품 물품", "용역 결과물", "시공 목적물", "정비 성과물"]

N_TPL = [
    "계약상대자가 납기를 지체한 경우 지체일수 1일마다 계약금액의 {p}/1000에 해당하는 지체상금을 부과한다.",
    "지체상금은 불가항력 사유가 확인된 경우에 한하여 발주기관의 승인을 받아 면제할 수 있다.",
    "납기 연장은 발주기관이 불가항력 사유를 확인하고 서면으로 승인한 경우에만 인정한다.",
    "발주기관은 납품된 {o}을 {d}일 이내에 검사하고 합격한 후에 대금을 지급한다.",
    "계약상대자는 하도급을 하려는 경우 사전에 발주기관의 서면 승인을 받아야 한다.",
    "어느 한쪽이 계약을 해지하려면 사유를 명시해 {d}일 전에 서면으로 통지해야 한다.",
    "{o}에 하자가 발생하면 계약상대자는 보증 기간 안에 무상으로 보수하거나 교체한다.",
    "계약 이행 중 알게 된 비공개 사항은 제3자에게 누설하지 않으며 위반 시 책임을 진다.",
    "{o}에 대한 지식재산권의 귀속과 사용 범위는 별도 합의서에 따라 정한다.",
    "대금은 검사 합격 후 청구서를 접수한 날부터 {d}일 이내에 지급한다.",
    "계약금액의 조정은 물가변동 기준에 따라 양 당사자가 서면으로 합의한 경우에만 한다.",
    "계약상대자는 이행보증서를 계약 체결 시 제출하며 보증금은 계약 이행 완료 후 반환한다.",
    "납품 일정은 계약서 별표의 일자를 따르며 변경 시 서면 합의가 필요하다.",
    "지체상금은 면제 사유가 없는 한 부과하며 이의는 정해진 절차로 제기할 수 있다.",
    "검사에 합격하지 못한 {o}은 계약상대자가 교체하며 그 비용은 계약상대자가 부담한다.",
    "계약상대자의 귀책 사유로 이행이 불가능하면 발주기관은 서면으로 통지하고 계약을 해지할 수 있다.",
    "지연이 예상되면 계약상대자는 사유와 만회 계획을 발주기관에 서면으로 보고한다.",
    "보증 기간 중 발생한 결함은 계약상대자가 책임지고 보수한다.",
    "분쟁은 먼저 협의로 해결하고 협의가 되지 않으면 관할 법원의 판단에 따른다.",
]
T_TPL = [
    "계약상대자의 지체상금은 전액 면제하며 발주기관은 이에 대해 이의를 제기하지 못한다.",
    "납품 즉시 검사에 합격한 것으로 보며 발주기관은 이후 검사를 요구하지 않는다.",
    "계약상대자는 언제든지 일방적으로 계약을 해지할 수 있으며 그에 따른 손해는 발주기관이 부담한다.",
    "{o}의 추가 비용은 사유를 불문하고 발주기관이 전액 부담하며 별도 협의 없이 청구할 수 있다.",
    "{o}에 관한 지식재산권은 모두 계약상대자에게 귀속되고 발주기관은 사용할 수 없다.",
    "하도급은 신고 없이 제한 없이 허용하며 발주기관은 하도급 내용을 확인하지 않는다.",
    "대금은 검사 전에 전액 선지급하며 발주기관은 반환 청구권을 포기한다.",
    "계약 이행 중 알게 된 사항의 누설로 인한 책임을 계약상대자는 지지 않는다.",
    "하자 보수 책임은 인도 후 즉시 소멸하며 발주기관은 보수를 요구할 수 없다.",
    "발주기관은 계약상대자가 청구하는 금액을 거절할 수 없으며 거절하면 위약금을 지급한다.",
    "분쟁이 생기면 계약상대자가 정하는 중재 기관에서만 해결하고 발주기관은 따른다.",
    "발주기관의 검사는 생략하고 계약상대자의 시험 결과서로 대신한다.",
    "견적 금액은 사후 정산 없이 그대로 확정하고 증빙 자료는 받지 않는다.",
]
D_TPL = [
    "납기는 자재 수급 사정에 따라 계약상대자가 연장할 수 있다.",
    "납품 일정은 가능한 한 지키도록 노력하며 지연되어도 사유를 보고하지 않는다.",
    "공급 지연이나 선행 업체 사정도 불가항력으로 보아 납기를 자동 연장한다.",
    "착수일은 선행 업체의 납품이 끝난 뒤 별도로 정하며 완료 기한은 정하지 않는다.",
    "납기일 이후 1회에 한해 기한 없이 자동 연장되는 것으로 한다.",
    "납기는 계약상대자의 생산 일정이 확정되는 대로 통보한다.",
    "납품 시점은 계약상대자 사정이 허락하는 때로 하고 날짜는 따로 정하지 않는다.",
    "자재가 늦게 들어오면 완료 일정은 그만큼 자동으로 늦춰진다.",
    "기한 내에 인도하지 못해도 계약상대자에게 책임을 묻지 않는다.",
    "일정이 늦어져도 계약상대자는 새 계획서를 낼 필요가 없다.",
]
# 학습에 쓰지 않는 '처음 보는 표현'(모델이 문장 틀을 외웠는지 확인하는 용도)
HOLD = {
    "정상": ["검사에 불합격한 경우 계약상대자는 지정 기한 안에 재납품하며 재검사 비용은 상대자가 부담한다.",
           "계약상대자가 법령을 위반하여 계약을 이행하지 못하면 발주기관은 서면 통지 후 계약을 해지할 수 있다.",
           "납기가 지연될 우려가 있으면 상대자는 즉시 사유와 만회 계획을 서면으로 보고해야 한다.",
           "하자담보 책임 기간은 인수일부터 기산하며 기간 중 발생한 결함은 상대자가 책임진다."],
    "독소": ["발주기관은 계약상대자가 요구하는 대금을 이유 없이 거절하지 못하며 거절 시 위약금을 낸다.",
           "계약 내용에 관한 분쟁은 계약상대자가 지정하는 기관에서만 다루며 발주기관은 이에 따른다.",
           "발주기관의 검수 권한은 포기하고 계약상대자의 자체 시험 성적서로 갈음한다.",
           "계약상대자가 제출한 견적은 사후 정산 없이 확정 단가로 하며 증빙은 요구하지 않는다."],
    "납기위험": ["납품 시기는 상대자의 형편이 허락하는 시점으로 하고 별도 날짜는 정하지 않는다.",
             "원자재 확보가 늦어지면 완료 일자는 자동으로 뒤로 미뤄지는 것으로 본다.",
             "인도 기한은 참고용이며 지키지 못해도 책임을 묻지 않는다.",
             "일정 지연이 있어도 상대자는 변경 계획을 제출할 의무가 없다."],
}
SUFFIX = ["", "", "", " 세부 사항은 시행 세칙에 따른다.", " 이 조항은 계약 전체에 적용한다.", " 필요한 경우 양 당사자가 협의한다.", " 관련 서식은 별지를 따른다."]
SWAP = [("발주기관", "계약담당관"), ("계약상대자", "업체")]


def make_clause(rng, cls, hold=False, idx=None):
    """조항 한 문장 생성. 표현 일부는 동의어로 바꾸고 접미 문구를 무작위로 붙인다(idx를 주면 그 번호의 문장 틀을 사용)."""
    if idx is not None:
        tpl = {"정상": N_TPL, "독소": T_TPL, "납기위험": D_TPL}[cls]
        s = tpl[idx].format(p=rng.choice(["1.0", "1.5", "2.0"]), d=int(rng.choice([7, 10, 14, 30])), o=rng.choice(OBJ))
    elif hold:
        s = HOLD[cls][rng.integers(len(HOLD[cls]))]
    else:
        tpl = {"정상": N_TPL, "독소": T_TPL, "납기위험": D_TPL}[cls]
        s = tpl[rng.integers(len(tpl))].format(p=rng.choice(["1.0", "1.5", "2.0"]), d=int(rng.choice([7, 10, 14, 30])), o=rng.choice(OBJ))
    if rng.random() < 0.3:
        a, b = SWAP[rng.integers(2)]
        s = s.replace(a, b)
    return s + SUFFIX[rng.integers(len(SUFFIX))]


def make_contracts(n=200, seed=7):
    """가상 계약 n건(속성 16개 + 조항 6개 + 규정 위반 정답)을 만든다."""
    rng = np.random.default_rng(seed)
    sl = rng.beta(1.2, 3.2, n)                                  # 계약서 작성이 허술한 정도(잠재 변수)
    ctype = rng.choice(TYPES, n, p=[.5, .3, .2])
    amount = np.clip(np.exp(rng.normal(np.log(220), 1.1, n)), 5, 5000).round(0)
    method = np.where(rng.random(n) < .06 + .3 * sl, "수의계약", rng.choice(METHODS[:3], n, p=[.5, .3, .2]))
    period = np.array([rng.integers(60, 300) if t == "물품" else rng.integers(90, 400) if t == "용역" else rng.integers(180, 720) for t in ctype])
    delivery = (period * rng.uniform(.5, 1.0, n)).astype(int)
    bad_logic = rng.random(n) < .015 * (1 + 3 * sl)
    delivery = np.where(bad_logic, period + rng.integers(10, 60, n), delivery)
    pen = np.where(rng.random(n) < .22 * sl, rng.choice([.3, .5, .8], n), rng.choice([1.0, 1.5, 2.0, 2.5], n))
    adv = np.where(rng.random(n) < .22 * sl, rng.choice([50, 60, 70], n), rng.choice([0, 10, 20, 30, 40], n))
    bond = np.where(rng.random(n) < .25 * sl + .5 * (adv > 40), rng.choice([0, 5], n), rng.choice([10, 10, 15, 20], n))
    war = np.array([rng.choice([3, 6]) if (rng.random() < .25 * s and t == "물품") else rng.choice([12, 24, 36]) if t == "물품" else rng.choice([0, 6, 12]) if t == "용역" else rng.choice([24, 36, 60]) for t, s in zip(ctype, sl)])
    insp = np.where(rng.random(n) < .3 * sl, rng.choice([21, 30], n), rng.choice([5, 7, 10, 14], n))
    sub = np.where(rng.random(n) < .15 * sl, rng.choice([60, 70, 80], n), rng.choice([0, 10, 20, 30, 40, 50], n))
    padj = rng.random(n) < .75 - .4 * sl
    term = rng.random(n) > .015 * (1 + 6 * sl)
    sec = rng.random(n) > .015 * (1 + 6 * sl)
    ip = rng.random(n) > .03 * (1 + 4 * sl)
    clauses, cls_true = [], []
    for i in range(n):
        pt, pd_ = .01 + .08 * sl[i], .02 + .08 * sl[i]
        cl = [rng.choice(CLASSES, p=[1 - pt - pd_, pt, pd_]) for _ in range(6)]
        cls_true.append(cl); clauses.append([make_clause(rng, c) for c in cl])
    df = pd.DataFrame({"contract_id": [f"C-{i + 1:04d}" for i in range(n)], "ctype": ctype, "method": method, "amount_m": amount,
                       "period_days": period, "delivery_days": delivery, "penalty_permil": pen, "advance_pct": adv, "perf_bond_pct": bond,
                       "warranty_months": war, "inspection_days": insp, "subcontract_pct": sub, "price_adjust": padj,
                       "termination": term, "security": sec, "ip_clause": ip})
    hard = ((pen < 1.0) | ((adv > 40) & (bond < 10)) | ((amount >= 300) & (method == "수의계약")) | ((ctype == "물품") & (war < 12))
            | ~term | ~sec | (sub > 50) | (delivery > period))
    toxic = np.array([c.count("독소") for c in cls_true])
    delay = np.array([c.count("납기위험") for c in cls_true])
    split = (method == "수의계약") & (amount >= 250) & (amount < 300) & (rng.random(n) < .6)      # 계약 쪼개기 의심(임계값 바로 아래)
    df["clauses"], df["clause_cls"] = clauses, cls_true
    df["n_toxic"], df["n_delay"] = toxic, delay
    viol = hard | (toxic > 0) | split | (delay >= 2)
    waived = hard & (toxic == 0) & (rng.random(n) < .12)                  # 계약관이 예외 사유를 인정해 위반으로 보지 않은 건(자료의 현실적 잡음)
    hidden = ~viol & (rng.random(n) < .03 + .08 * sl)                       # 속성·문구에 드러나지 않는 위반(별첨 합의 등)
    df["violation"] = (viol & ~waived) | hidden
    df["vtype"] = np.where(hidden, "기타", np.where(hard, "속성위반", np.where(toxic > 0, "독소조항", np.where(split | (delay >= 2), "기타", "-"))))
    return df

SIMPLE_KW = {"독소": ["면제", "일방", "포기", "전액", "누설", "하도급", "선지급"], "납기위험": ["연장", "자동", "노력", "통보", "기한"]}

# ---- 조항 문장 분류(정상 / 독소 / 납기위험): 글자 n-gram TF-IDF + 로지스틱 회귀 ----
def clause_set(seed, n, hold=False, mix=(.55, .25, .20)):
    """조항 문장 n개와 정답 분류를 만든다. hold=True면 학습에 쓰지 않은 '처음 보는 표현'으로 만든다."""
    # [TODO : 학생 실습 4] LLM/자연어처리 기반 독소 조항, 납기 지체 위험 문구 자동 식별 모델 고도화 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def fit_clause_model(seed=100, n=1200):
    """학습용 조항 문장으로 분류 모델을 학습한다."""
    # [TODO : 학생 실습 4] LLM/자연어처리 기반 독소 조항, 납기 지체 위험 문구 자동 식별 모델 고도화 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def keyword_pred(s):
    """비교용 기준선: 단순 키워드(부정 표현·문맥을 보지 않음) 적중으로만 분류."""
    if any(k in s for k in SIMPLE_KW["독소"]):
        return "독소"
    return "납기위험" if any(k in s for k in SIMPLE_KW["납기위험"]) else "정상"

# [교육용가상자료] 표준 문안 라이브러리(수정 권고용, 실제 표준계약서 조문이 아님)
STD = {"지체상금": "납기를 지체하면 지체일수마다 계약금액의 1/1000 이상에 해당하는 지체상금을 부과하며 면제는 불가항력 사유가 확인되고 서면 승인을 받은 경우에 한다.",
       "검사": "발주기관은 납품된 물품을 정해진 기간 안에 검사하고 합격한 후에 대금을 지급하며 검사 권한은 생략하거나 대체할 수 없다.",
       "해지": "계약 해지는 사유를 명시하고 정해진 기간 전에 서면으로 통지해야 하며 책임 있는 쪽이 손해를 부담한다.",
       "추가비용": "추가 비용은 사전에 협의해 서면으로 합의한 범위에서만 인정하며 증빙을 갖추어 청구한다.",
       "지식재산": "지식재산권의 귀속과 사용 범위는 별도 합의서로 정하며 발주기관은 목적물을 사용할 수 있다.",
       "하도급": "하도급은 사전에 발주기관의 서면 승인을 받아야 하며 하도급 내용은 확인할 수 있어야 한다.",
       "대금": "대금은 검사 합격 후 청구를 받아 정해진 기간 안에 지급하고 선지급은 보증을 갖춘 범위에서만 한다.",
       "보안": "계약 이행 중 알게 된 비공개 사항은 누설하지 않으며 위반 시 책임을 진다.",
       "하자": "보증 기간 중 발생한 결함은 계약상대자가 책임지고 무상으로 보수하거나 교체한다.",
       "분쟁": "분쟁은 먼저 협의로 해결하고 협의가 되지 않으면 관할 법원의 판단에 따른다.",
       "납기": "납기는 계약서에 날짜로 명시하고 변경은 불가항력 사유 확인 후 서면 승인으로만 한다."}
TOPIC_T = ["지체상금", "검사", "해지", "추가비용", "지식재산", "하도급", "대금", "보안", "하자", "대금", "분쟁", "검사", "추가비용"]      # T_TPL 순서와 대응

model = fit_clause_model()
Xa, ya = clause_set(200, 600)                    # 학습과 같은 문장 틀(새로 뽑은 문장)
Xb, yb = clause_set(201, 300, hold=True)         # 학습에 없는 처음 보는 표현
print(f"[데이터] 학습 조항 1,200개 / 시험 A(같은 문장 틀) {len(Xa)}개 / 시험 B(처음 보는 표현) {len(Xb)}개 / 분류 3종: {', '.join(CLASSES)}")

pa, pb = model.predict(Xa), model.predict(Xb)
acc_a, acc_b = np.mean(pa == np.array(ya)), np.mean(pb == np.array(yb))
X_all, y_all, p_all = Xa + Xb, np.array(ya + yb), np.concatenate([pa, pb])
acc = np.mean(p_all == y_all)
kb = np.array([keyword_pred(s) for s in X_all])
print(f"\n[분류 정확도] 시험 A {acc_a:.1%}, 시험 B(처음 보는 표현) {acc_b:.1%}, 전체 {acc:.1%} (목표 85% 이상: {'달성' if acc >= 0.85 else '미달'})")
print(f"단순 키워드 기준선 전체 {np.mean(kb == y_all):.1%} (부정·문맥을 보지 못해 '면제 사유가 없는 한 부과' 같은 정상 문장을 오탐)")
cm = pd.DataFrame(confusion_matrix(y_all, p_all, labels=CLASSES), index=[f"실제 {c}" for c in CLASSES], columns=[f"예측 {c}" for c in CLASSES])
print(cm.to_string())
risk_idx = y_all != "정상"
rec_ = np.mean(p_all[risk_idx] != "정상")
print(f"위험 조항(독소·납기위험) 포착률 {rec_:.1%}, 정상 문장을 위험으로 오탐한 비율 {np.mean(p_all[~risk_idx] != '정상'):.1%}")

# 오분류 사례
print("\n[오분류 사례(시험 B)]")
shown = 0
for s, t, p in zip(Xb, yb, pb):
    if t != p and shown < 3:
        shown += 1; print(f"  실제 {t} / 예측 {p}: {s[:48]}...")
if shown == 0:
    print("  없음")

# 근거 표현: 해당 분류에 가장 크게 기여한 글자 조각
vec, lr = model[0], model[-1]
names = np.array(vec.get_feature_names_out())


def evidence(text, cls, k=3):
    x = vec.transform([text])
    contrib = x.multiply(lr.coef_[list(lr.classes_).index(cls)]).toarray()[0]
    return [names[j].strip() for j in np.argsort(-contrib)[:k] if contrib[j] > 0]


# 표준 문안 Top-3 추천(과거 처리 완료 사례와의 일치도): 독소 조항 120개와 납기 위험 문구 60개
rng = np.random.default_rng(5)
sv = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)).fit(list(STD.values()) + X_all)
SM = sv.transform(list(STD.values()))
topics = list(STD)
tests = [(make_clause(rng, "독소", idx=i), TOPIC_T[i]) for i in rng.integers(0, len(T_TPL), 120)] + [(make_clause(rng, "납기위험"), "납기") for _ in range(60)]
jn = topics.index("납기")


def similarity(s):
    """문안 유사도. 분류 결과가 납기위험이면 납기 문안을 우선하고, 독소 조항이면 납기 문안은 제외한다."""
    raw = cosine_similarity(sv.transform([s]), SM)[0]
    sc = raw.copy()
    sc[jn] += 1.0 if model.predict([s])[0] == "납기위험" else -1.0
    return sc, raw


hit1 = hit3 = 0
for s, t in tests:
    sc, _ = similarity(s)
    order = np.argsort(-sc)[:3]
    hit1 += topics[order[0]] == t; hit3 += t in [topics[j] for j in order]
print(f"\n[표준 문안 추천 일치도] 위험 조항 {len(tests)}건: Top-1 {hit1 / len(tests):.1%}, Top-3 {hit3 / len(tests):.1%} (과거 처리된 수정 문안 주제와 일치 기준)")
print("\n[위험 조항별 근거와 수정 권고 Top-3 예시]")
for s, t in [tests[0], tests[7], tests[-1]]:
    pred = model.predict([s])[0]; prob = model.predict_proba([s])[0].max()
    sc, sim = similarity(s)
    order = np.argsort(-sc)[:3]
    print(f"  조항: {s[:46]}...")
    print(f"    분류 {pred}(확신도 {prob:.2f}), 근거 표현 {evidence(s, pred) or '-'}")
    for r, j in enumerate(order, 1):
        print(f"    권고 {r}순위 [{topics[j]}] 유사도 {sim[j]:.2f}: {list(STD.values())[j][:30]}...")
print("※ 모델은 후보를 표시하고 수정 문안은 참고용이다. 불공정 조항 여부와 문안 확정은 계약관·법무 담당자가 판단한다.")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(["keyword", "model A", "model B", "model all"], [np.mean(kb == y_all), acc_a, acc_b, acc], color=["gray", "tab:blue", "tab:orange", "tab:green"])
ax[0].axhline(.85, color="r", ls="--"); ax[0].set_ylim(0, 1.05); ax[0].set_title("Clause classification accuracy")
im = ax[1].imshow(cm.to_numpy(), cmap="Blues"); ax[1].set_xticks(range(3), ["ok", "toxic", "delay"]); ax[1].set_yticks(range(3), ["ok", "toxic", "delay"]); ax[1].set_title("Confusion matrix (all test)")
for i in range(3):
    for j in range(3):
        ax[1].text(j, i, int(cm.to_numpy()[i, j]), ha="center", va="center")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 04 예상 실행 결과 (Output)</b></summary>

```text
[데이터] 학습 조항 1,200개 / 시험 A(같은 문장 틀) 600개 / 시험 B(처음 보는 표현) 300개 / 분류 3종: 정상, 독소, 납기위험

[분류 정확도] 시험 A 100.0%, 시험 B(처음 보는 표현) 95.3%, 전체 98.4% (목표 85% 이상: 달성)
단순 키워드 기준선 전체 64.4% (부정·문맥을 보지 못해 '면제 사유가 없는 한 부과' 같은 정상 문장을 오탐)
         예측 정상  예측 독소  예측 납기위험
실제 정상      504      0        0
실제 독소        0    222        0
실제 납기위험     14      0      160
위험 조항(독소·납기위험) 포착률 96.5%, 정상 문장을 위험으로 오탐한 비율 0.0%

[오분류 사례(시험 B)]
  실제 납기위험 / 예측 정상: 일정 지연이 있어도 상대자는 변경 계획을 제출할 의무가 없다. 필요한 경우 양 당사자가...
  실제 납기위험 / 예측 정상: 일정 지연이 있어도 상대자는 변경 계획을 제출할 의무가 없다. 이 조항은 계약 전체에 ...
  실제 납기위험 / 예측 정상: 일정 지연이 있어도 상대자는 변경 계획을 제출할 의무가 없다....

[표준 문안 추천 일치도] 위험 조항 180건: Top-1 94.4%, Top-3 97.8% (과거 처리된 수정 문안 주제와 일치 기준)

[위험 조항별 근거와 수정 권고 Top-3 예시]
  조항: 하자 보수 책임은 인도 후 즉시 소멸하며 계약담당관은 보수를 요구할 수 없다....
    분류 독소(확신도 0.99), 근거 표현 ['관은', '관은', '즉시']
    권고 1순위 [하자] 유사도 0.12: 보증 기간 중 발생한 결함은 계약상대자가 책임지고 무상...
    권고 2순위 [검사] 유사도 0.09: 발주기관은 납품된 물품을 정해진 기간 안에 검사하고 합...
    권고 3순위 [하도급] 유사도 0.06: 하도급은 사전에 발주기관의 서면 승인을 받아야 하며 하...
  조항: 용역 결과물의 추가 비용은 사유를 불문하고 발주기관이 전액 부담하며 별도 협의 없이...
    분류 독소(확신도 0.99), 근거 표현 ['전액', '액', '전액']
    권고 1순위 [추가비용] 유사도 0.16: 추가 비용은 사전에 협의해 서면으로 합의한 범위에서만 ...
    권고 2순위 [지식재산] 유사도 0.12: 지식재산권의 귀속과 사용 범위는 별도 합의서로 정하며 ...
    권고 3순위 [해지] 유사도 0.12: 계약 해지는 사유를 명시하고 정해진 기간 전에 서면으로...
  조항: 납품 일정은 가능한 한 지키도록 노력하며 지연되어도 사유를 보고하지 않는다. 세부 ...
    분류 납기위험(확신도 0.99), 근거 표현 ['일정', '일정', '도']
    권고 1순위 [납기] 유사도 0.03: 납기는 계약서에 날짜로 명시하고 변경은 불가항력 사유 ...
    권고 2순위 [보안] 유사도 0.08: 계약 이행 중 알게 된 비공개 사항은 누설하지 않으며 ...
    권고 3순위 [해지] 유사도 0.06: 계약 해지는 사유를 명시하고 정해진 기간 전에 서면으로...
※ 모델은 후보를 표시하고 수정 문안은 참고용이다. 불공정 조항 여부와 문안 확정은 계약관·법무 담당자가 판단한다.
```
</details>
